# Trace Count v16_2: three-character sets in Tiny Shakespeare

This isolated revision prepares a reproducible pool of distinct three-character
sets, mixes raw-language and counting-formatted training examples through
`TASK_OCCURRENCE_RATIO`, and evaluates fixed raw/task/mixture suites on train,
held-out validation, and final-only test regions.

## 1. Mount Google Drive first

In [ ]:
from pathlib import Path

# Keep the uploaded source and result paths derived from one editable root.
DRIVE_REPO_ROOT = Path(
    "/content/drive/MyDrive/Colab Notebooks/NIAH_synthetic/"
    "Synthetic_CoT_NiaH_Count-main"
)
DRIVE_RESULTS_ROOT = DRIVE_REPO_ROOT / "colab_results"
DRIVE_READY = False
if Path("/content").exists():
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    DRIVE_READY = True
    print("Drive mounted; source expected at", DRIVE_REPO_ROOT)
else:
    print("Local runtime: Drive mount skipped")

## 2. Repository and environment

In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

setup_started = time.perf_counter()
assert DRIVE_READY, "Run the Google Drive mount cell before environment setup"
assert DRIVE_REPO_ROOT.exists(), DRIVE_REPO_ROOT
assert (DRIVE_REPO_ROOT / "pyproject.toml").exists(), (
    f"pyproject.toml not found under {DRIVE_REPO_ROOT}"
)
assert DRIVE_RESULTS_ROOT == DRIVE_REPO_ROOT / "colab_results"
DRIVE_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# Copy source to the Colab VM: training against Drive is substantially slower.
LOCAL_REPO_ROOT = Path("/content/Synthetic_CoT_NiaH_Count-main")
shutil.copytree(
    DRIVE_REPO_ROOT,
    LOCAL_REPO_ROOT,
    dirs_exist_ok=True,
    ignore=shutil.ignore_patterns(
        ".git",
        ".venv*",
        "__pycache__",
        ".pytest_cache",
        ".ruff_cache",
        ".mypy_cache",
        "*.egg-info",
        "runs",
        "artifacts",
        "colab_results",
    ),
)

repo = LOCAL_REPO_ROOT
os.chdir(repo)

# Validate Colab's preinstalled binary stack without modifying it. If this
# fails, discard the runtime rather than mixing binary package generations.
scientific_probe = subprocess.run(
    [
        sys.executable,
        "-c",
        "import matplotlib,numpy,pandas,seaborn,torch,tqdm",
    ],
    capture_output=True,
    text=True,
)
if scientific_probe.returncode:
    print(scientific_probe.stdout)
    print(scientific_probe.stderr)
    raise RuntimeError(
        "Colab scientific-package imports are inconsistent. Use Runtime > "
        "Disconnect and delete runtime, reconnect, and rerun from the first cell."
    )

# Preserve Colab's binary-compatible NumPy/pandas/scientific stack.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."],
    check=True,
)

# Editable-install .pth files are processed only when Python starts. Make the
# copied src-layout package importable in this already-running notebook kernel.
src_root = str(repo / "src")
if src_root not in sys.path:
    sys.path.insert(0, src_root)

import synthetic_counting_v16_2
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, Markdown, display

package_path = Path(synthetic_counting_v16_2.__file__).resolve()
assert package_path.is_relative_to(Path(src_root)), (
    f"Notebook kernel imported stale package from {package_path}"
)
subprocess_package_path = Path(
    subprocess.check_output(
        [
            sys.executable,
            "-c",
            "import synthetic_counting_v16_2 as p; print(p.__file__)",
        ],
        text=True,
    ).strip()
).resolve()
assert subprocess_package_path.is_relative_to(Path(src_root)), (
    f"Subprocess imported stale package from {subprocess_package_path}"
)

corpus_path = (
    repo
    / "src"
    / "synthetic_counting_v11"
    / "resources"
    / "tiny_shakespeare"
    / "input.txt"
)
assert corpus_path.exists(), f"Tiny Shakespeare is missing: {corpus_path}"

print({
    "drive_repo": str(DRIVE_REPO_ROOT),
    "working_repo": str(repo),
    "src_root": src_root,
    "kernel_package": str(package_path),
    "subprocess_package": str(subprocess_package_path),
    "corpus": str(corpus_path),
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "torch": torch.__version__,
    "cuda": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
})
print(f"Environment setup block: {time.perf_counter() - setup_started:.1f} seconds")

In [ ]:
RUN_TESTS = False  # optional developer preflight; not required for training
if RUN_TESTS:
    test_process = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "tests/test_synthetic_counting_v16_2.py"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(test_process.stdout, end="")
    test_process.check_returncode()
else:
    print("Optional repository tests skipped; pipeline validation remains enabled.")

## 3. Easy-to-edit settings

In [ ]:
VERSION = "v16_2"
PRESET = "main"                 # use "debug" for an end-to-end check
SEED = 1234
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TASK_OCCURRENCE_RATIO = 1        # 0 = raw Shakespeare; 1 = all counting tasks
COUNT_MAX_THRESHOLD = 30
WEIGHT_DECAY = 0.01              # AdamW decay on all trainable parameters; 0.0 disables it
FINAL_COUNT_LOSS_WEIGHT = 1.0    # >1 upweights the final numeric answer target
COT_TRACE_LOSS_WEIGHT = 1.0      # >1 upweights CoT trace indices and marker characters
RPE_max_update = True            # True uses max_render_len - 1; False keeps legacy max_relative_distance=256
RUN_ROPE_NONTHINKING = False
RUN_ROPE_THINKING = False
RUN_RPE_NONTHINKING = True
RUN_RPE_THINKING = True
MAX_TRAIN_STEPS = 10_000          # optimizer steps for each enabled model
MAX_STEPS_FOR_LANGUAGE_PRED = 1_500  # through this step use all-token LM loss; afterward train task output only
CHECKPOINT_EVERY_STEPS = 500      # save step 0, every N steps, the objective boundary, and the final step
EVAL_EXAMPLES_PER_COUNT = 50     # examples for each count; suite size = this value x COUNT_MAX_THRESHOLD
NEEDLE_POOL_SIZE = 100
NEEDLE_POOL_FREQUENCY_THRESHOLD = 0.12
OUT_ROOT = "runs/synthetic_counting_v16_2"
RUN_NAME = None                  # default name records all important settings
SKIP_COMPLETED = True
CHECKPOINT_SYNC_ROOT = (
    DRIVE_RESULTS_ROOT / "v16_2_live_checkpoints" if DRIVE_READY else None
)

ENABLED_MODEL_VARIANTS = tuple(
    variant
    for enabled, variant in (
        (RUN_ROPE_NONTHINKING, "rope/nonthinking"),
        (RUN_ROPE_THINKING, "rope/thinking"),
        (RUN_RPE_NONTHINKING, "rpe/nonthinking"),
        (RUN_RPE_THINKING, "rpe/thinking"),
    )
    if enabled
)
if not ENABLED_MODEL_VARIANTS:
    raise ValueError("Enable at least one of the four model variants")

from synthetic_counting_v16_2.config import preset_config
PLANNED_CONFIG = preset_config(
    PRESET,
    seed=SEED,
    device=DEVICE,
    task_occurrence_ratio=TASK_OCCURRENCE_RATIO,
    count_max_threshold=COUNT_MAX_THRESHOLD,
    weight_decay=WEIGHT_DECAY,
    final_count_loss_weight=FINAL_COUNT_LOSS_WEIGHT,
    cot_trace_loss_weight=COT_TRACE_LOSS_WEIGHT,
    rpe_max_update=RPE_max_update,
    enabled_model_variants=ENABLED_MODEL_VARIANTS,
    train_steps=MAX_TRAIN_STEPS,
    max_steps_for_language_pred=MAX_STEPS_FOR_LANGUAGE_PRED,
    checkpoint_every=CHECKPOINT_EVERY_STEPS,
    eval_examples_per_count=EVAL_EXAMPLES_PER_COUNT,
    needle_pool_size=NEEDLE_POOL_SIZE,
    needle_pool_frequency_threshold=NEEDLE_POOL_FREQUENCY_THRESHOLD,
)
EVAL_EXAMPLES_PER_SUITE = EVAL_EXAMPLES_PER_COUNT * COUNT_MAX_THRESHOLD
PERIODIC_TF_EXAMPLES_PER_MODEL = 7 * EVAL_EXAMPLES_PER_SUITE
LANGUAGE_PREDICTION_STEPS = min(MAX_STEPS_FOR_LANGUAGE_PRED, MAX_TRAIN_STEPS)
TASK_OUTPUT_ONLY_STEPS = max(0, MAX_TRAIN_STEPS - MAX_STEPS_FOR_LANGUAGE_PRED)
PLANNED_CHECKPOINT_STEPS = sorted({
    0,
    *range(CHECKPOINT_EVERY_STEPS, MAX_TRAIN_STEPS + 1, CHECKPOINT_EVERY_STEPS),
    min(MAX_STEPS_FOR_LANGUAGE_PRED, MAX_TRAIN_STEPS),
    MAX_TRAIN_STEPS,
})
print({
    "config": PLANNED_CONFIG.to_dict(),
    "enabled_model_variants": ENABLED_MODEL_VARIANTS,
    "number_of_models": len(ENABLED_MODEL_VARIANTS),
    "weight_decay": WEIGHT_DECAY,
    "rpe_max_update": RPE_max_update,
    "max_relative_distance": PLANNED_CONFIG.max_relative_distance,
    "language_prediction_steps": LANGUAGE_PREDICTION_STEPS,
    "task_output_only_steps": TASK_OUTPUT_ONLY_STEPS,
    "task_output_starts": {"nonthinking": "<Ans>", "thinking": "<Think>"},
    "max_steps_per_model": MAX_TRAIN_STEPS,
    "total_planned_optimizer_steps": len(ENABLED_MODEL_VARIANTS) * MAX_TRAIN_STEPS,
    "checkpoint_every_steps": CHECKPOINT_EVERY_STEPS,
    "planned_checkpoint_steps_per_model": PLANNED_CHECKPOINT_STEPS,
    "numeric_checkpoints_per_model": len(PLANNED_CHECKPOINT_STEPS),
    "eval_examples_per_suite": EVAL_EXAMPLES_PER_SUITE,
    "periodic_teacher_forced_examples_per_model": PERIODIC_TF_EXAMPLES_PER_MODEL,
})

## 4. Prepare split, pool, and fixed evaluation suites

In [ ]:
base_cmd = [
    sys.executable, "-u", "-m", "synthetic_counting_v16_2.run_v16_2",
    "--preset", PRESET,
    "--device", DEVICE,
    "--seed", str(SEED),
    "--task-occurrence-ratio", str(TASK_OCCURRENCE_RATIO),
    "--count-max-threshold", str(COUNT_MAX_THRESHOLD),
    "--weight-decay", str(WEIGHT_DECAY),
    "--final-count-loss-weight", str(FINAL_COUNT_LOSS_WEIGHT),
    "--cot-trace-loss-weight", str(COT_TRACE_LOSS_WEIGHT),
    "--train-steps", str(MAX_TRAIN_STEPS),
    "--max-steps-for-language-pred", str(MAX_STEPS_FOR_LANGUAGE_PRED),
    "--checkpoint-every", str(CHECKPOINT_EVERY_STEPS),
    "--eval-examples-per-count", str(EVAL_EXAMPLES_PER_COUNT),
    "--needle-pool-size", str(NEEDLE_POOL_SIZE),
    "--needle-pool-frequency-threshold", str(NEEDLE_POOL_FREQUENCY_THRESHOLD),
    "--out-root", OUT_ROOT,
]
base_cmd.append("--rpe-max-update" if RPE_max_update else "--no-rpe-max-update")
for variant in ENABLED_MODEL_VARIANTS:
    base_cmd += ["--model-variant", variant]
if RUN_NAME is not None:
    base_cmd += ["--run-name", RUN_NAME]
if CHECKPOINT_SYNC_ROOT is not None:
    base_cmd += ["--checkpoint-sync-root", str(CHECKPOINT_SYNC_ROOT)]
if SKIP_COMPLETED:
    base_cmd.append("--skip-completed")
prepare_started = time.perf_counter()
subprocess.run([*base_cmd, "--stage", "prepare"], check=True)
print(f"Prepare block: {time.perf_counter() - prepare_started:.1f} seconds")

from synthetic_counting_v16_2.config import default_run_name
RUN_DIR = Path(OUT_ROOT) / (RUN_NAME or default_run_name(PLANNED_CONFIG))
display(pd.read_csv(RUN_DIR / "tables" / "needle_pool.csv"))
display(Image(filename=str(RUN_DIR / "figures" / "needle_pool_frequency_distribution.png")))

## 5. Train, analyze, and plot

In [ ]:
training_started = time.perf_counter()
subprocess.run([*base_cmd, "--stage", "train,attention,state,plots"], check=True)
print(f"Training/final-diagnostics block: {time.perf_counter() - training_started:.1f} seconds")
print("RUN_DIR =", RUN_DIR.resolve())

## 6. Analyze internal mechanisms across saved checkpoints

In [ ]:
# All metric families are independently switchable. Defaults use fixed,
# count-balanced examples at every checkpoint for comparable trajectories.
RUN_CHECKPOINT_DYNAMICS = True
RUN_ATTENTION_DYNAMICS = True
RUN_HIDDEN_STATE_DYNAMICS = True
RUN_GENERATED_TRACE_DYNAMICS = True
RUN_COUNTERFACTUAL_DYNAMICS = True
RUN_REPRESENTATION_STABILITY = True
FORCE_CHECKPOINT_DYNAMICS = False  # False resumes completed checkpoint partitions

DYNAMICS_ATTENTION_EXAMPLES_PER_COUNT = 20  # 10 select heads + 10 held-out reporting examples
DYNAMICS_AR_EXAMPLES_PER_COUNT = 10         # autoregressive generations per true count
DYNAMICS_STATE_TRAIN_EXAMPLES_PER_COUNT = 40  # ridge/centroid fitting examples per count
DYNAMICS_STATE_EVAL_EXAMPLES_PER_COUNT = 15   # held-out state examples per count

if RUN_CHECKPOINT_DYNAMICS:
    from synthetic_counting_v16_2.training import checkpoint_steps

    checkpoint_inventory = {
        variant: [step for step, _ in checkpoint_steps(RUN_DIR, *variant.split("/"))]
        for variant in ENABLED_MODEL_VARIANTS
    }
    print("Checkpoint inventory:", checkpoint_inventory)
    required_steps = {min(MAX_STEPS_FOR_LANGUAGE_PRED, MAX_TRAIN_STEPS), MAX_TRAIN_STEPS}
    for variant, available_steps in checkpoint_inventory.items():
        missing = required_steps - set(available_steps)
        if missing:
            raise FileNotFoundError(
                f"{variant} is missing required checkpoint step(s) {sorted(missing)}; "
                "restore the run's checkpoints from Google Drive before analysis"
            )
    dynamics_cmd = [
        sys.executable, "-u", "scripts/analyze_v16_2_checkpoint_dynamics.py",
        str(RUN_DIR),
        "--device", DEVICE,
        "--attention-examples-per-count", str(DYNAMICS_ATTENTION_EXAMPLES_PER_COUNT),
        "--ar-examples-per-count", str(DYNAMICS_AR_EXAMPLES_PER_COUNT),
        "--state-train-examples-per-count", str(DYNAMICS_STATE_TRAIN_EXAMPLES_PER_COUNT),
        "--state-eval-examples-per-count", str(DYNAMICS_STATE_EVAL_EXAMPLES_PER_COUNT),
    ]
    for enabled, flag in (
        (RUN_ATTENTION_DYNAMICS, "--skip-attention"),
        (RUN_HIDDEN_STATE_DYNAMICS, "--skip-states"),
        (RUN_GENERATED_TRACE_DYNAMICS, "--skip-generated"),
        (RUN_COUNTERFACTUAL_DYNAMICS, "--skip-counterfactual"),
        (RUN_REPRESENTATION_STABILITY, "--skip-similarity"),
    ):
        if not enabled:
            dynamics_cmd.append(flag)
    if FORCE_CHECKPOINT_DYNAMICS:
        dynamics_cmd.append("--force")
    dynamics_started = time.perf_counter()
    subprocess.run(dynamics_cmd, check=True)
    print(f"Checkpoint-dynamics block: {time.perf_counter() - dynamics_started:.1f} seconds")
    print("Dynamics manifest:", RUN_DIR / "analysis" / "checkpoint_dynamics" / "manifest.json")
    print("Detailed tables:", RUN_DIR / "tables" / "checkpoint_*.csv")
    for figure_name in (
        "checkpoint_mechanism_overview.png",
        "checkpoint_attention_retrieval_emergence.png",
        "checkpoint_final_count_probe_heatmap.png",
        "checkpoint_counterfactual_trace_readout.png",
        "runtime_breakdown.png",
    ):
        figure_path = RUN_DIR / "figures" / figure_name
        if figure_path.exists():
            display(Image(filename=str(figure_path)))
else:
    print("Checkpoint-dynamics analysis skipped; saved checkpoints remain available.")

## 7. Inspect train-versus-held-out loss curves

In [ ]:
inspect_started = time.perf_counter()
losses = pd.read_csv(RUN_DIR / "tables" / "eval_loss_curves.csv")
display(losses)
display(Image(filename=str(RUN_DIR / "figures" / "learning_loss_suites_train_vs_heldout.png")))
test_summary = RUN_DIR / "tables" / "test_loss_summary.csv"
if test_summary.exists():
    display(Markdown("**Final-only untouched test results**"))
    display(pd.read_csv(test_summary))
print(f"Result-display block: {time.perf_counter() - inspect_started:.1f} seconds")

## 8. Save the complete result bundle

In [ ]:
import shutil
from datetime import datetime

save_started = time.perf_counter()
if DRIVE_READY:
    destination = DRIVE_RESULTS_ROOT / f"{RUN_DIR.name}_{datetime.now():%Y%m%d_%H%M%S}"
    shutil.copytree(RUN_DIR, destination, dirs_exist_ok=True)
    print("Saved:", destination)
else:
    print("Drive unavailable; results remain at", RUN_DIR.resolve())
print(f"Result-copy block: {time.perf_counter() - save_started:.1f} seconds")